Used Car Price Prediction

Dataset contains important features about used cars sold on cardekho.com in India

Task: To predict the price of the car to be sold based on its features given

In [126]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [127]:
df = pd.read_csv('/content/cardekho_dataset.csv',index_col=[0])

In [128]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15411 entries, 0 to 19543
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   car_name           15411 non-null  object 
 1   brand              15411 non-null  object 
 2   model              15411 non-null  object 
 3   vehicle_age        15411 non-null  int64  
 4   km_driven          15411 non-null  int64  
 5   seller_type        15411 non-null  object 
 6   fuel_type          15411 non-null  object 
 7   transmission_type  15411 non-null  object 
 8   mileage            15411 non-null  float64
 9   engine             15411 non-null  int64  
 10  max_power          15411 non-null  float64
 11  seats              15411 non-null  int64  
 12  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 1.6+ MB


Feature Engineering

1.Data Cleaning

In [ ]:
#Handle missing values
df.isnull().sum()

,0
car_name,0
brand,0
model,0
vehicle_age,0
km_driven,0
seller_type,0
fuel_type,0
transmission_type,0
mileage,0
engine,0


In [129]:
#Remove unnecessary columns like car name and brand

df.drop(['car_name','brand'],axis=1,inplace=True)
df

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...
19537,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
19540,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
19541,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
19542,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [130]:
#Finding no. of diff types of features

num_features = [feature for feature in df.select_dtypes(include='number').columns]
print("No. of Numerical features=",len(num_features))

cat_features = [feature for feature in df.select_dtypes(exclude='number').columns]
print("No. of Categorical features=",len(cat_features))

discrete_features = [feature for feature in df.select_dtypes(include='number').columns if len(df[feature].unique())<=25]
print("No. of Discrete features=",len(discrete_features))

continuous_features = [feature for feature in df.select_dtypes(include='number').columns if feature not in discrete_features]
print("No. of Continuous features=",len(continuous_features))

No. of Numerical features= 7
No. of Categorical features= 4
No. of Discrete features= 2
No. of Continuous features= 5


In [170]:
#Independent and dependent features

X = df.drop('selling_price',axis=1)
y = df['selling_price']

Feature Encoding

In [171]:
len(df['model'].unique())

120

In [172]:
#label Encoding the 'model' feature of dataset coz it has too many categories

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [173]:
num_features = X.select_dtypes(include='number').columns
onehot_columns = ['fuel_type','seller_type','transmission_type']

In [174]:
#OneHotEncoding with ColumnTransformer -- seller_type, fuel_type, transmission_type

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler

ct = ColumnTransformer(transformers=
      [('encoder',OneHotEncoder(drop='first'),onehot_columns),
      ('scaler',StandardScaler(),num_features)
      ],remainder='passthrough')


In [136]:
ct

ColumnTransformer(remainder='passthrough',
                  transformers=[('encoder', OneHotEncoder(drop='first'),
                                 ['fuel_type', 'seller_type',
                                  'transmission_type']),
                                ('scaler', StandardScaler(),
                                 Index(['model', 'vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power',
       'seats'],
      dtype='object'))])

In [138]:
onehot_columns

['fuel_type', 'seller_type', 'transmission_type']

In [175]:
X= ct.fit_transform(X)

In [176]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,0.0,0.0,0.0,1.0,1.0,0.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,0.0,0.0,0.0,1.0,1.0,0.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,1.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [177]:
#train test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [180]:
y_train

,selling_price
14238,1825000
1731,515000
13218,7500000
403,435000
13550,200000
...,...
6581,665000
17029,249000
6839,250000
1104,620000


Model Training and Model Selection

In [181]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,root_mean_squared_error

In [183]:
#Function to evaluate performances

def evaluate_model(true,predicted):
  mae = mean_absolute_error(true,predicted)
  mse = mean_squared_error(true,predicted)
  r2 = r2_score(true,predicted)
  return mae, mse, r2

In [184]:
#Beginning model training with 6 diff types of models

models = {
    "Linear Regression":LinearRegression(),   #taking default model parameters for all
    "Ridge":Ridge(),
    "Lasso":Lasso(),
    "KNN":KNeighborsRegressor(),
    "Decision Tree":DecisionTreeRegressor(),
    "Random Forest":RandomForestRegressor()
}
for val in models.values():
  model = val
  model.fit(X_train,y_train)

  #Model predictions
  y_test_pred = model.predict(X_test)
  y_train_pred = model.predict(X_train)

  #Model Evaluation on Training data
  print("\n",model)
  print("Training data Performance:")
  train_mae, train_mse, train_r2 = evaluate_model(y_train,y_train_pred)
  print(f"  MAE: {train_mae:.4f}")
  print(f"  MSE: {train_mse:.4f}")
  print(f"  R2 Score: {train_r2:.4f}")

  #Model evaluation on Test data
  print("\nTest data Performance:")
  test_mae, test_mse, test_r2 = evaluate_model(y_test,y_test_pred)
  print(f"  MAE: {test_mae:.4f}")
  print(f"  MSE: {test_mse:.4f}")
  print(f"  R2 Score: {test_r2:.4f}")



 LinearRegression()
Training data Performance:
  MAE: 268101.6071
  MSE: 306756099359.7596
  R2 Score: 0.6218

Test data Performance:
  MAE: 279618.5794
  MSE: 252550062888.5655
  R2 Score: 0.6645

 Ridge()
Training data Performance:
  MAE: 268059.8015
  MSE: 306756818740.9266
  R2 Score: 0.6218

Test data Performance:
  MAE: 279557.2169
  MSE: 252540243247.9686
  R2 Score: 0.6645

 Lasso()
Training data Performance:
  MAE: 268099.2227
  MSE: 306756104248.3573
  R2 Score: 0.6218

Test data Performance:
  MAE: 279614.7461
  MSE: 252549134830.3719
  R2 Score: 0.6645

 KNeighborsRegressor()
Training data Performance:
  MAE: 91425.6652
  MSE: 106201478306.2946
  R2 Score: 0.8691

Test data Performance:
  MAE: 112671.9186
  MSE: 64046693647.6241
  R2 Score: 0.9149

 DecisionTreeRegressor()
Training data Performance:
  MAE: 5164.8199
  MSE: 432524990.5364
  R2 Score: 0.9995

Test data Performance:
  MAE: 125626.4650
  MSE: 94728278209.1082
  R2 Score: 0.8742

 RandomForestRegressor()
Traini

We choose knn and RandomForest Regressor as they have best r2 scores

In [185]:
#Initialize parameters for HyperParameter Tuning

knn_params = {'n_neighbors': [2,3,10,20,40,50]}
rf_params = {'n_estimators': [100,200,500,1000],
              'max_depth': [5,8,15,None,10],
              'min_samples_split': [2,8,15,20],
              'max_features': [5,7,"auto",8]
             }

In [186]:
#Model list for HyperParameter Tuning
randomcv_models = [('KNN',KNeighborsRegressor(),knn_params),
                    ('Random Forest',RandomForestRegressor(),rf_params)
                  ]


In [187]:
#randomSearchCV

from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name,model,params in randomcv_models:
  random = RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=100,cv=3,verbose=2,n_jobs=-1)
  random.fit(X_train,y_train)

  model_param[name] = random.best_params_

for model_name in model_param:
  print(f"Best parameters for {model_name} are {model_param[model_name]}")

Fitting 3 folds for each of 6 candidates, totalling 18 fits


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=100. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 100 candidates, totalling 300 fits


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
78 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.11/dist-packages/sklearn/utils/

Best parameters for KNN are {'n_neighbors': 10}
Best parameters for Random Forest are {'n_estimators': 200, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}


In [188]:
#Using best params for both chosen models to get the performances

models = {
    "KNN":KNeighborsRegressor(n_neighbors=10,n_jobs=-1),
    "Random Forest":RandomForestRegressor(n_estimators=100,max_depth=15,min_samples_split=2,max_features=8,n_jobs=-1)
}
for val in models.values():
  model = val
  model.fit(X_train,y_train)

  #Model predictions
  y_test_pred = model.predict(X_test)
  y_train_pred = model.predict(X_train)

  #Model Evaluation on Training data
  print("\n",model)
  print("Training data Performance:")
  train_mae, train_mse, train_r2 = evaluate_model(y_train,y_train_pred)
  print(f"  MAE: {train_mae:.4f}")
  print(f"  MSE: {train_mse:.4f}")
  print(f"  R2 Score: {train_r2:.4f}")

  #Model evaluation on Test data
  print("\nTest data Performance:")
  test_mae, test_mse, test_r2 = evaluate_model(y_test,y_test_pred)
  print(f"  MAE: {test_mae:.4f}")
  print(f"  MSE: {test_mse:.4f}")
  print(f"  R2 Score: {test_r2:.4f}")



 KNeighborsRegressor(n_jobs=-1, n_neighbors=10)
Training data Performance:
  MAE: 103491.4909
  MSE: 132107689408.6632
  R2 Score: 0.8371

Test data Performance:
  MAE: 117479.7032
  MSE: 69616775253.2030
  R2 Score: 0.9075

 RandomForestRegressor(max_depth=15, max_features=8, n_jobs=-1)
Training data Performance:
  MAE: 54721.6412
  MSE: 18447862732.3321
  R2 Score: 0.9773

Test data Performance:
  MAE: 98036.8499
  MSE: 45755845550.3003
  R2 Score: 0.9392
